In [1]:
# ============================================================
# FEATURE ENGINEERING & PREPROCESSING PIPELINE
# Customer Churn Prediction
# ============================================================

# 1. Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# 2. Load Dataset
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())


# 3. Basic Information
print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

print(df.isnull().sum())


# 4. Data Cleaning

# Remove customer ID because it is not useful for prediction
df.drop("customerID", axis=1, inplace=True)

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Convert Churn target to 0 and 1
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})


# 5. Separate Features and Target
X = df.drop("Churn", axis=1)
y = df["Churn"]

print("\nFeatures Shape:", X.shape)
print("Target Shape:", y.shape)


# 6. Train-Test Split
# IMPORTANT: Split before preprocessing to prevent data leakage

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)


# 7. Identify Numerical and Categorical Features

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)


# 8. Numerical Preprocessing Pipeline
# Missing values -> Median
# Scaling -> StandardScaler

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# 9. Categorical Preprocessing Pipeline
# Missing values -> Most Frequent
# Encoding -> One-Hot Encoding

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# 10. ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)


# 11. Complete Machine Learning Pipeline

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    ))
])


# 12. Train Model

print("\n" + "=" * 60)
print("TRAINING MODEL")
print("=" * 60)

model_pipeline.fit(X_train, y_train)

print("Model training completed successfully!")


# 13. Make Predictions

y_pred = model_pipeline.predict(X_test)


# 14. Model Evaluation

accuracy = accuracy_score(y_test, y_pred)

print("\n" + "=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# 15. Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


# 16. Correlation Analysis

print("\n" + "=" * 60)
print("CORRELATION ANALYSIS")
print("=" * 60)

correlation = X_train[numerical_features].corr()

plt.figure(figsize=(10, 6))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Numerical Feature Correlation Heatmap")
plt.show()


# 17. Feature Importance using Mutual Information

print("\n" + "=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

# Transform training data
X_train_processed = model_pipeline.named_steps[
    "preprocessor"
].transform(X_train)

# Get feature names after preprocessing
feature_names = model_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()


# Identify categorical encoded features
discrete_mask = np.array([
    any(name.startswith(f"categorical__{col}_")
        for col in categorical_features)
    for name in feature_names
])


# Calculate Mutual Information
mi_scores = mutual_info_classif(
    X_train_processed,
    y_train,
    discrete_features=discrete_mask,
    random_state=42
)


# Create feature importance table
mi_df = pd.DataFrame({
    "Feature": feature_names,
    "MI Score": mi_scores
})

mi_df = mi_df.sort_values(
    by="MI Score",
    ascending=False
)

print("\nTop 15 Important Features:")
print(mi_df.head(15))


# 18. Feature Importance Visualization

plt.figure(figsize=(10, 6))

sns.barplot(
    data=mi_df.head(15),
    x="MI Score",
    y="Feature"
)

plt.title("Top 15 Features Using Mutual Information")
plt.xlabel("Mutual Information Score")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()


# 19. Save Complete Pipeline

joblib.dump(
    model_pipeline,
    "customer_churn_preprocessing_pipeline.pkl"
)

print("\n" + "=" * 60)
print("PIPELINE SAVED")
print("=" * 60)

print("File created:")
print("customer_churn_preprocessing_pipeline.pkl")


# 20. Test Saved Pipeline

loaded_pipeline = joblib.load(
    "customer_churn_preprocessing_pipeline.pkl"
)

test_predictions = loaded_pipeline.predict(
    X_test.head(5)
)

print("\nPredictions from Saved Pipeline:")
print(test_predictions)


# ============================================================
# PROJECT COMPLETED
# ============================================================

print("\n" + "=" * 60)
print("PROJECT COMPLETED SUCCESSFULLY!")
print("=" * 60)

print("""
The project includes:

1. Dataset loading
2. Missing value handling
3. Train/Test splitting
4. Numerical feature scaling
5. Categorical One-Hot Encoding
6. ColumnTransformer
7. Scikit-Learn Pipeline
8. Random Forest Classification
9. Correlation Analysis
10. Mutual Information Feature Importance
11. Model Evaluation
12. Reusable Pipeline Saving
""")

FileNotFoundError: [Errno 2] No such file or directory: 'WA_Fn-UseC_-Telco-Customer-Churn.csv'